#  추천 시스템 - ETF 추천 로직

1. LLM과 RAG를 결합한 추천 시스템의 아키텍처 이해
2. Text2SQL과 하이브리드 검색을 활용한 ETF 검색 시스템 구현
3. LangGraph를 활용한 다단계 워크플로우 설계 방법 습득
4. 구조화된 출력(Structured Output)을 통한 일관된 결과 생성 학습
5. Gradio를 이용한 AI 애플리케이션 배포 실습

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import warnings
warnings.filterwarnings('ignore')

`(3) SQLite DB`

In [ ]:
from langchain_community.utilities import SQLDatabase

# ETF 데이터베이스 연결
db = SQLDatabase.from_uri(
    "sqlite:///etf_database.db",
    ignore_tables=['ETFsInfo']
    )

# 사용 가능한 테이블 목록 
print(db.dialect)
print(db.get_usable_table_names())
etfs = db.run("SELECT * FROM ETFs LIMIT 5;")

for etf in eval(etfs):
    print(etf)

`(4) SQL QA Chain`

-  DB 테이블에서 고유명사 추출

In [ ]:
import ast
import re

def query_as_list(db, query):
    res = db.run(query)
    res = [el for sub in ast.literal_eval(res) for el in sub if el]
    res = [re.sub(r"\b\d+\b", "", string).strip() for string in res]
    return list(set(res))


etfs = query_as_list(db, "SELECT DISTINCT 종목명 FROM ETFs")
fund_managers = query_as_list(db, "SELECT DISTINCT 운용사 FROM ETFs")
underlying_assets = query_as_list(db, "SELECT DISTINCT 기초지수 FROM ETFs")

print("ETFs:", len(etfs))
print("Fund Managers:", len(fund_managers))
print("Underlying Assets:", len(underlying_assets))

print(etfs[:5])
print(fund_managers[:5])
print(underlying_assets[:5])

- 고유명사를 벡터스토어에 저장

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 임베딩 벡터 저장소 생성
vector_store = InMemoryVectorStore(embeddings)

# ETF 종목명과 운용사를 임베딩 벡터로 변환
_ = vector_store.add_texts(etfs + fund_managers + underlying_assets)
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

- 고유명사를 키워드 검색으로 적용

In [ ]:
### kiwipiepy 토크나이저 함수 정의

from kiwipiepy import Kiwi
import re

# Kiwi 토크나이저 초기화
kiwi = Kiwi()

def korean_tokenizer(text):
    """kiwipiepy를 사용한 한국어 토크나이징 함수"""
    # 기본 전처리
    text = re.sub(r'[^\w\s]', '', text)  # 특수문자 제거
    text = text.lower()  # 소문자 변환
    
    # Kiwi로 형태소 분석
    tokens = kiwi.tokenize(text)
    
    # 명사, 형용사, 동사, 외국어, 한자 등 필터링 (선별 추출)
    filtered_tokens = []
    for token in tokens:
        if token.tag in ['NNG', 'NNP', 'VA', 'VV', 'SL', 'SH']:  # 일반명사, 고유명사, 형용사, 동사, 외국어, 한자
            filtered_tokens.append(token.form)
    
    return filtered_tokens if filtered_tokens else [token.form for token in tokens]

# 테스트
print(korean_tokenizer("다우존스 관련 ETF는 무엇인가요?"))


In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# 고유명사를 Document 객체로 변환
proper_noun_docs = [Document(page_content=name) for name in etfs + fund_managers + underlying_assets]

# BM25 검색기 생성 (Kiwi 토크나이저 적용)
bm25_retriever = BM25Retriever.from_documents(
    proper_noun_docs,
    preprocess_func=korean_tokenizer,
    k=20,
)
print(f"BM25 검색기 생성 완료: {len(proper_noun_docs)}개 문서 등록")

# 테스트
query = "다우존스 관련 ETF는 무엇인가요?"
results = bm25_retriever.invoke(query)
print(f"\nQuery: {query}")
print("-" * 40)
for result in results:
    print(f"Document: {result.page_content}")

In [ ]:
### 하이브리드 앙상블 리트리버 생성
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

# 테스트
query = "다우존스 관련 ETF는 무엇인가요?"
results = ensemble_retriever.invoke(query)
print(f"Query: {query}")
print("-" * 40)
for result in results:
    print(f"Document: {result.page_content}")

- 검색 결과 변환하는 함수

In [ ]:
from langchain_core.tools import tool

# 검색 도구 생성 - 하이브리드 검색기 사용
@tool("search_proper_nouns")
def entity_retriever_tool(query: str) -> str:
    """
    Use to look up values to filter on. Input is an approximate spelling 
    of the proper noun, output is valid proper nouns. Use the noun most 
    similar to the search.
    """
    docs = ensemble_retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

# 검색 도구 테스트
print(entity_retriever_tool.invoke("다우존스 관련 ETF는 무엇인가요?"))

- 에이전트 정의

In [ ]:
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import QuerySQLDatabaseTool
from langgraph.graph import START, StateGraph 
from IPython.display import Image, display

# 상태 정보를 저장하는 State 클래스
class State(TypedDict):
    question: str  # 입력 질문
    query: str     # 생성된 쿼리
    result: str    # 쿼리 결과
    answer: str    # 생성된 답변

# llm 모델 생성
llm = ChatOpenAI(model="gpt-4.1-mini")

# sql 쿼리 생성 프롬프트 
template = """
Given an input question, create a syntactically correct {dialect} query to run to help find the answer. Unless the user specifies in his question a specific number of examples they wish to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.

Never query for all the columns from a specific table, only ask for a the few relevant columns given the question.

Pay attention to use only the column names that you can see in the schema description. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

Only use the following tables:
{table_info}

Entity names and their relationships to consider:
{entity_info}

## Matching Guidelines
- Use exact matches when comparing entity names
- Check for historical name variations if available
- Apply case-sensitive matching for official names
- Handle both Korean and English entity names when present

Question: {input}
"""
query_prompt_template = ChatPromptTemplate.from_template(template)

# SQL 쿼리 생성 함수
class QueryOutput(TypedDict):
    """Generated SQL query."""
    query: Annotated[str, ..., "Syntactically valid SQL query."]

def write_query(state: State):
    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "dialect": db.dialect,
            "top_k": 10,
            "table_info": db.get_table_info(),
            "input": state["question"],
            "entity_info": entity_retriever_tool.invoke(state["question"]),
        }
    )
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}

# SQL 쿼리 실행 함수
def execute_query(state: State):
    """Execute SQL query."""
    execute_query_tool = QuerySQLDatabaseTool(db=db)
    return {"result": execute_query_tool.invoke(state["query"])}

# 답변 생성 함수
def generate_answer(state: State):
    """Answer question using retrieved information as context."""
    prompt = (
        "Given the following user question, corresponding SQL query, "
        "and SQL result, answer the user question.\n\n"
        f'Question: {state["question"]}\n'
        f'SQL Query: {state["query"]}\n'
        f'SQL Result: {state["result"]}'
    )
    response = llm.invoke(prompt)
    return {"answer": response.content}


# 상태 그래프 생성
graph_builder = StateGraph(State)

graph_builder.add_node("write_query", write_query)
graph_builder.add_node("execute_query", execute_query)
graph_builder.add_node("generate_answer", generate_answer)

graph_builder.add_edge(START, "write_query")
graph_builder.add_edge("write_query", "execute_query")
graph_builder.add_edge("execute_query", "generate_answer")  
graph = graph_builder.compile()

# 상태 그래프 시각화
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# 고유명사를 사용하는 쿼리 테스트 - 운용사
for step in graph.stream(
    {"question": "케이비에서 운용하는 헬스케어 ETF는 무엇인가요?"}, stream_mode="updates"
):
    print(step)

In [ ]:
# 고유명사를 사용하는 쿼리 테스트 - 기초지수, 펀드명
for step in graph.stream(
    {"question": "다우존스 관련 ETF를 찾아주세요."}, stream_mode="updates"
):
    print(step)

## **LLM과 RAG를 결합한 추천 시스템** 

- **전통적인 추천** 시스템의 주요 문제점들:
    - **데이터 희소성** 문제로 인해 대부분의 사용자-아이템 상호작용 정보가 부족함
    - 신규 사용자나 아이템에 대한 **콜드 스타트** 현상으로 초기 추천이 어려움
    - **명시적 피드백**(평점, 리뷰)의 수집이 제한적이며 신뢰성에 한계가 있음 
    - **틈새 아이템**과 신규 콘텐츠는 충분한 상호작용 데이터 확보가 어려워 추천이 제한됨 

- **RAG 기반 추천** 시스템의 가능성:

    - **벡터 데이터베이스**와 **LLM**을 결합하여 효율적인 검색과 자연어 처리를 구현
    - **LangChain** 기반 파이프라인으로 상품 데이터의 벡터화부터 추천까지 일관된 프로세스를 구축함
    - **RAG 시스템**을 통해 컨텍스트를 고려한 개인화된 추천이 가능함
    - 전통적 추천 시스템의 한계를 **자연어 기반 추천**으로 보완하여 더 풍부한 설명을 제공 가능

### 1) **전체 시스템 개요**

- **상태 관리 시스템**이 사용자 프로필부터 추천 결과까지 체계적으로 데이터를 관리

- 각 단계별 추천 정확도를 높일 수 있도록 세분화된 **프로세스 체인**으로 역할을 구분

- **구조화된 프롬프트**를 통해 일관된 형식의 결과물을 생성

- **출력 데이터**가 JSON 형식으로 표준화되어 후속 처리에 활용 

```mermaid
flowchart LR
    Q[사용자 질문] --> P[프로필 분석]
    P -->|투자 성향 & 목표 추출| S[SQL 쿼리 생성]
    S -->|데이터베이스 조회| E[후보 ETF 검색]
    E -->|필터링 조건 적용| R[랭킹 및 필터링]
    R -->|상위 후보 선정| D[설명 생성]
    
    style Q fill:#e1f5fe,stroke:#01579b
    style P fill:#e8f5e9,stroke:#1b5e20
    style S fill:#fff3e0,stroke:#e65100
    style E fill:#f3e5f5,stroke:#4a148c
    style R fill:#fbe9e7,stroke:#bf360c
    style D fill:#e8eaf6,stroke:#1a237e

    subgraph 데이터 처리
    S
    E
    R
    end

    subgraph 출력 생성
    D
    end
```

### 2) **상태(State) 관리**

- **State** 클래스는 **TypedDict**를 상속받아 ETF 추천 시스템의 상태를 관리함
- 사용자와의 상호작용 정보를 **question**과 **user_profile**에 저장
- ETF 분석 결과는 **candidates**와 **rankings** 리스트에 보관
- 최종 출력 관련 데이터는 **explanation**과 **final_answer**에 기록

In [ ]:
from typing import TypedDict

# 상태 정보를 저장하는 State 클래스
class State(TypedDict):
    question: str          # 사용자 입력 질문
    user_profile: dict     # 사용자 프로필 정보
    query: str             # 생성된 SQL 쿼리
    candidates: list       # 후보 ETF 목록
    rankings: list         # 순위가 매겨진 ETF 목록
    explanation: str       # 추천 이유 설명
    final_answer: str      # 최종 추천 답변

### 3) **프로세스 체인**

- 시스템은 **사용자 입력**으로 시작하여 **프로필 분석**을 통해 맞춤형 추천을 준비함
- **SQL 쿼리**와 **ETF 검색** 과정을 통해 데이터베이스에서 적절한 상품을 찾음
- **랭킹 시스템**을 통해 최적의 ETF를 선별하고 **맞춤형 설명**을 생성함

`(1) 사용자 프로필 분석`

- 시스템은 사용자의 **투자 성향**과 **위험 선호도**를 먼저 파악함
- **투자 목표**, **투자 기간**, **투자 가능 금액**을 종합적으로 분석
- 사용자의 **과거 투자 이력**과 **현재 포트폴리오** 구성을 검토
- 프로필 분석은 **개인화된 ETF 추천**을 위한 핵심 기초 데이터를 제공함


In [ ]:
from enum import Enum
from typing import List
from pydantic import BaseModel, Field

class RiskTolerance(Enum):
    CONSERVATIVE = "conservative"
    MODERATE = "moderate" 
    AGGRESSIVE = "aggressive"

class InvestmentHorizon(Enum):
    SHORT = "short"
    MEDIUM = "medium"
    LONG = "long"

class InvestmentProfile(BaseModel):
    risk_tolerance: RiskTolerance = Field(
        description="투자자의 위험 성향 (conservative/moderate/aggressive)"
    )
    investment_horizon: InvestmentHorizon = Field(
        description="투자 기간 (short/medium/long)"
    )
    investment_goal: str = Field(
        description="투자의 주요 목적 설명"
    )
    preferred_sectors: List[str] = Field(
        description="선호하는 투자 섹터 목록"
    )
    excluded_sectors: List[str] = Field(
        description="투자를 원하지 않는 섹터 목록"
    )
    monthly_investment: int = Field(
        description="월 투자 가능 금액 (원)"
    )

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 사용자 프로필 분석 프롬프트
PROFILE_TEMPLATE= """
사용자의 질문을 분석하여 투자 프로필을 생성해주세요.

사용자 질문: {question}
"""

profile_prompt = ChatPromptTemplate.from_template(PROFILE_TEMPLATE)

# 사용자 프로필 분석 모델 생성
llm = ChatOpenAI(model="gpt-4.1-mini")
profile_llm = llm.with_structured_output(InvestmentProfile)

# 사용자 프로필 분석 함수
def analyze_profile(state: State) -> dict:
    """사용자 질문을 분석하여 투자 프로필 생성"""
    prompt = profile_prompt.invoke({"question": state["question"]})
    response = profile_llm.invoke(prompt)
    return {"user_profile": dict(response)}


# 예시 질문 
question = """
저는 30대 초반의 직장인입니다. 
월 100만원 정도를 3년 이상 장기 투자하고 싶고,
기술 섹터와 헬스케어에 관심이 있습니다.
중위험 중수익을 추구하며, ESG 요소도 고려하고 싶습니다.
적합한 ETF를 추천해주세요.
"""

# 사용자 프로필 분석
result_profile = analyze_profile({"question": question})

# 결과 출력
result_profile

`(2) ETF 검색 쿼리 프롬프트 생성`

- **사용자 프로필** {user_profile}과 **입력 질문** {input}에 따라 최적화된 쿼리 생성

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.prompts import ChatPromptTemplate

# SQL Query Generation Template
QUERY_TEMPLATE = """
Given an input question and investment profile, create a syntactically correct {dialect} query to run. Unless specified, limit your query to at most {top_k} results. Order the results by most relevant columns based on the investment profile.

Never query for all columns from a specific table, only ask for relevant columns given the question and investment criteria.

Pay attention to use only the column names you can see in the schema description. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

Available tables:
{table_info}

Entity relationships:
{entity_info}

## Matching Guidelines
- Use exact matches when comparing entity names
- Check for historical name variations if available
- Apply case-sensitive matching for official names
- Handle both Korean and English entity names when present

Investment Profile:
{user_profile}

Question: {input}

## Constraints
1. Use only existing columns
2. Query only necessary columns (no SELECT *)
3. Follow correct table relationships
4. Consider performance and indexing
"""

# SQL Query Generation Prompt Template
query_prompt_template = ChatPromptTemplate.from_template(QUERY_TEMPLATE)

# SQL Query Output
class QueryOutput(TypedDict):
    """Generated SQL query."""
    query: Annotated[str, ..., "Syntactically valid SQL query."]
    explanation: Annotated[str, ..., "Query explanation and selection criteria (in 한국어)"]


def write_query(state: State):
    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "dialect": db.dialect,
            "top_k": 10,
            "table_info": db.get_table_info(),
            "input": state["question"],
            "entity_info": entity_retriever_tool.invoke(state["question"]),
            "user_profile": state["user_profile"],
        }
    )
    structured_llm = ChatOpenAI(model="gpt-4.1").with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"], "explanation": result["explanation"]}

# 테스트
result_query = write_query({ 
    "question": question,
    "user_profile": result_profile["user_profile"]
})

# 결과 출력
result_query


`(3) ETF 검색 쿼리 실행`

In [ ]:
from langchain_community.tools import QuerySQLDatabaseTool

def execute_query(state: State) -> dict:
    """SQL 쿼리 실행하여 후보 ETF 검색"""
    try:
        execute_query_tool = QuerySQLDatabaseTool(db=db)
        results = execute_query_tool.invoke(state["query"])
        if not results or results == "[]":
            return {"candidates": "검색 결과가 없습니다."}
        return {"candidates": results}
    except Exception as e:
        return {"candidates": f"쿼리 실행 중 오류 발생: {str(e)}"}

# 테스트
result_candidates = execute_query({
    "query": result_query["query"]
})

# 결과 출력
result_candidates

In [ ]:
for candidate in eval(result_candidates["candidates"]):
    print(candidate)

`(4) 후보 ETF 랭킹 및 필터링`

- **랭킹 시스템**은 **수익률**, **순자산총액**, **총보수**, **변동성** 등 5가지 핵심 지표를 평가함
- 순위 결정에는 **사용자 프로필** {user_profile}과 **후보 ETF** {candidates} 정보가 활용됨
- 각 ETF는 **점수화**되어 상위 3개가 출력됨

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.prompts import ChatPromptTemplate

RANKING_TEMPLATE = """
Rank the following ETF candidates based on the user's investment profile and return the top 3(three) ETFs.
Consider these factors when ranking:

1. 수익률
2. 변동성
3. 순자산총액
4. 총보수
5. User Profile matching score

User Profile:
{user_profile}

Candidate ETFs:
{candidates}

Table Info:
(table_info)
"""

# ETF Ranking Prompt Template
ranking_prompt = ChatPromptTemplate.from_template(RANKING_TEMPLATE)


# ETF Ranking Output
class ETFRanking(TypedDict):
    """Individual ETF ranking result"""
    rank: Annotated[int, ..., "Ranking position (1-5)"]
    etf_code: Annotated[str, ..., "ETF 종목코드 (6-digit)"]
    etf_name: Annotated[str, ..., "ETF 종목명"]
    score: Annotated[float, ..., "Composite score (0-100)"]
    ranking_reason: Annotated[str, ..., "Explanation for the ranking (in 한국어)"]

class ETFRankingResult(TypedDict):
    """Ranked ETFs"""
    rankings: List[ETFRanking]

# ETF Ranking Function
def rank_etfs(state: State) -> dict:
    """Rank ETF candidates based on user's investment profile"""
    prompt = ranking_prompt.invoke(
        {
            "user_profile": state["user_profile"],
            "candidates": state["candidates"],
            "table_info": db.get_table_info(),
        }
    )
    structured_llm = ChatOpenAI(model="gpt-4.1").with_structured_output(ETFRankingResult)
    results = structured_llm.invoke(prompt)

    return {"rankings": results}

# 테스트
result_rankings = rank_etfs({
    "user_profile": result_profile["user_profile"],
    "candidates": result_candidates["candidates"]
})  # type: ignore

# 결과 출력
result_rankings

`(5) 추천 설명 생성`

- ETF 설명은 **주요 특징{rankings}** 과 **사용자 적합성{user_profile}** 을 중심으로 구성
- **포트폴리오 구성안**을 제시하여 **실용적이고 이해하기 쉬운** 실질적인 투자 가이드를 제공
- **리스크 요소**와 **주의사항**을 명확히 설명하여 투자자 보호를 강화

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from decimal import Decimal
from langchain_core.prompts import ChatPromptTemplate

EXPLANATION_TEMPLATE = """
Please provide a comprehensive explanation for the recommended ETFs based on the user's investment profile.


[RECOMMENDATION EXPLANATION (Examples)]
1. ETF Characteristics
   - Investment strategy and approach
   - Historical performance overview
   - Fee structure and efficiency
   - Underlying assets and diversification

2. Profile Fit Analysis
   - Alignment with risk tolerance
   - Match with investment horizon
   - Sector preference compatibility
   - Investment goal contribution

3. Portfolio Construction
   - Recommended allocation percentages
   - Diversification benefits
   - Rebalancing considerations
   - Implementation strategy

4. Risk Considerations
   - Market risk factors
   - Specific ETF risks
   - Economic scenario impacts
   - Monitoring requirements

--------------------------------------------

[User Profile]
{user_profile}

[Selected ETFs]
{rankings}
"""

# 추천 설명 프롬프트
explanation_prompt = ChatPromptTemplate.from_template(EXPLANATION_TEMPLATE)


# 추천 설명 출력 스키마
class ETFRecommendation(BaseModel):
   """Individual ETF recommendation details"""
   etf_code: str = Field(..., description="ETF 종목코드 (6-digit)")
   etf_name: str = Field(..., description="ETF 종목명")
   allocation: Decimal = Field(..., description="Recommended allocation % (0-100)")
   description: str = Field(..., description="ETF description and investment strategy (in 한국어)")
   key_points: List[str] = Field(..., description="Key investment points (in 한국어)")
   risks: List[str] = Field(..., description="Risk factors to consider (in 한국어)")

class RecommendationExplanation(BaseModel):
   """ETF recommendation explanation with markdown formatting"""
   overview: str = Field(..., description="Overall strategy explanation (in 한국어)")
   recommendations: List[ETFRecommendation] = Field(..., description="ETF details")
   considerations: List[str] = Field(..., description="Important considerations (in 한국어)")
   
   # 마크다운 포맷으로 출력
   def to_markdown(self) -> str:
      """Convert explanation to markdown format"""
      markdown = [
            "# ETF 포트폴리오 추천",
            "",
            "## 투자 전략 개요",
            self.overview,
            "",
            "## 추천 ETF 포트폴리오",
            ""
      ]
      
      # 포트폴리오 구성 비율
      markdown.extend([
            "| ETF | 종목코드 | 추천비중 |",
            "|-----|----------|----------|"
      ])
      
      for rec in self.recommendations:
         markdown.append(
            f"| {rec.etf_name} | {rec.etf_code} | {rec.allocation}% |"
            )
      
      # ETF 상세 설명
      markdown.append("\n## ETF 상세 설명\n")
      
      for rec in self.recommendations:
         markdown.extend([
               f"### {rec.etf_name} ({rec.etf_code})",
               rec.description,
               "",
               "**주요 투자 포인트:**",
               "".join([f"\n* {point}" for point in rec.key_points]),
               "",
               "**투자 위험:**",
            "".join([f"\n* {risk}" for risk in rec.risks]),
         ""
         ])
      
      # 투자 리스크 고려사항
      markdown.extend([
            "## 투자 시 고려사항",
            "".join([f"\n* {item}" for item in self.considerations]),
            ""
      ])
      
      return "\n".join(markdown)
   

# 추천 설명 생성 함수
def generate_explanation(state: dict) -> dict:
   """Generate structured ETF recommendation explanation"""
   # 프롬프트 생성
   prompt = explanation_prompt.invoke({
      "rankings": state["rankings"],
      "user_profile": state["user_profile"]
   })
   
   # 구조화된 출력 생성
   structured_llm = ChatOpenAI(model="gpt-4.1-mini").with_structured_output(RecommendationExplanation)
   response = structured_llm.invoke(prompt)

   return {"final_answer": {
      "explanation": response.model_dump(), 
      "markdown": response.to_markdown()
   }}
   

# 테스트
result_explanation = generate_explanation({
   "rankings": result_rankings["rankings"],
   "user_profile": result_profile["user_profile"]
})

In [ ]:
# 결과 출력
result_explanation["final_answer"]["explanation"]

In [ ]:
# 마크다운 출력
from IPython.display import Markdown
Markdown(result_explanation["final_answer"]["markdown"])

`(6) 상태 그래프 생성`

- 전체 프로세스 체인을 연결하여 상태 그래프를 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

# 상태 그래프 생성
graph_builder = StateGraph(State)

graph_builder.add_node("analyze_profile", analyze_profile)
graph_builder.add_node("write_query", write_query)
graph_builder.add_node("execute_query", execute_query)
graph_builder.add_node("rank_etfs", rank_etfs)
graph_builder.add_node("generate_explanation", generate_explanation)

graph_builder.add_edge(START, "analyze_profile")
graph_builder.add_edge("analyze_profile", "write_query")
graph_builder.add_edge("write_query", "execute_query")
graph_builder.add_edge("execute_query", "rank_etfs")
graph_builder.add_edge("rank_etfs", "generate_explanation")
graph_builder.add_edge("generate_explanation", END)

graph = graph_builder.compile()


# 상태 그래프 시각화
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# 사용자 질문 예시

test_questions = [
    """
    40대 중반의 직장인입니다. 
    월 200만원 정도를 1년 이상 장기 투자하고 싶고,
    환율과 금리에 관심이 있습니다.
    저위험 안정적인 수익을 추구하며, ESG 요소도 고려하고 싶습니다.
    적합한 ETF를 추천해주세요.
    """,
    """
    20대 후반의 대학생입니다. 
    월 50만원 정도를 1년 이상 장기 투자하고 싶고,
    환율과 금리에 관심이 있습니다.
    고위험 고수익을 추구하며, ESG 요소도 고려하고 싶습니다.
    적합한 ETF를 추천해주세요.
    """,
    """
    32살 맞벌이 부부입니다. 
    둘이 합쳐서 월 200만원을 노후자금으로 굴리고 싶어요. 
    제약/바이오와 클라우드 컴퓨팅 쪽을 보고 있습니다. 
    안정성과 수익성의 균형을 맞추고 싶고, 
    지속가능한 기업들 위주로 투자하고 싶습니다.
    적합한 ETF를 추천해주세요.
    """
]


# 상태 그래프 실행
etf_recommendations = []
for question in test_questions:
    etf_recommendation = graph.invoke(
        {"question": question}
    )
    etf_recommendations.append(etf_recommendation)

In [ ]:
# 결과 출력
Markdown(etf_recommendations[0]["final_answer"]["markdown"])

In [ ]:
# 결과 출력
Markdown(etf_recommendations[1]["final_answer"]["markdown"])

In [ ]:
# 결과 출력
Markdown(etf_recommendations[2]["final_answer"]["markdown"])

---

## **Gradio 인터페이스 구현 및 Hugging Face Spaces 배포** 

#### 1. **사전 준비**
- [Hugging Face](https://huggingface.co/spaces) 계정 생성이 필요 (무료)
- 로컬에서 작동하는 Gradio 앱을 준비
- 새로운 **가상 환경**에서 실행 (crwal4ai 의존성과 gradio 최신 버전 문제)
- **app.py** 파일을 생성하여 프로젝트 폴더로 복사 (etf_database.db 파일, .env 파일도 함께 복사)

- pyproject.toml

- 터미널에서 다음 명령어 실행

- 서버 종료: 터미널에서 Ctrl + C

- requirements.txt 파일 생성 

- .gitignore 파일 생성

#### 2. **배포 방법 (터미널 이용)**
- Gradio 앱이 있는 디렉토리로 이동 

- 터미널에서 다음 명령어 실행

- CLI가 안내하는 대로 기본 메타데이터 입력
   - HF Token : Hugging Face 계정의 Access Token 복사해서 붙여넣고 엔터 
   - git credential 설정 (y/n) : n
   - Space 이름
   - Any Spaces secrets (y/n) : env 환경변수 설정 (OPENAI_API_KEY 붙여 넣고 엔터)
   - 또는 HF Spaces Secrets에 직접 추가 (OPENAI_API_KEY 값 입력 후 엔터)

- 배포 완료 후 제공되는 URL로 접근 가능

---

## [실습] ETF 추천 시스템 개선

### 필수 과제
1. **에러 처리 개선**: execute_query 함수에 더 상세한 에러 처리 로직 추가
2. **테스트 검증**: 추천 결과의 필수 필드 검증 로직 추가

### 선택 과제
1. **랭킹 알고리즘 개선**: 사용자 프로필에 따른 가중치 조정 로직 추가
2. **추가 필터링**: 총보수, 순자산총액 임계값 기반 필터링 추가
3. **프롬프트 개선**: Text2SQL 프롬프트의 정확도 향상
4. **프로세스 체인 추가**: 새로운 분석 단계 추가 (예: 포트폴리오 다각화 분석)